# Modeling: Baseline


## Purpose
Train logistic regression and naive baselines. Establish the performance floor that all advanced models must beat.

### Historical Benchmarks
- Naive (always pick higher seed): ~71% accuracy
- Logistic Regression: typically ~73-75%
- Good model target: >76% accuracy, log_loss < 0.52

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys; sys.path.insert(0, str(Path(".").parent / "src"))
from nba_predictor.config import cfg
from nba_predictor.models.baseline import run_cv_baseline, get_feature_cols
from nba_predictor.evaluation.cv_strategy import get_cv_fold_summary


## Load Data and Inspect CV Structure

In [ ]:
series_path = cfg.project_root / "data/processed/series_dataset.parquet"
if not series_path.exists():
    raise RuntimeError("Run make process first")
df = pd.read_parquet(series_path)
print(f"Dataset: {len(df)} series, seasons {df.season.min()}–{df.season.max()}")
print("\nCV fold structure:")
print(get_cv_fold_summary(df).to_string())


## Naive Baseline: Always Pick Higher Seed

In [ ]:
# Compute naive accuracy
naive_acc = df["higher_seed_wins"].mean()
print(f"Naive accuracy (always higher seed): {naive_acc:.3f}")
print(f"This is the floor — any model must beat {naive_acc:.1%}")


## Logistic Regression Walk-Forward CV

In [ ]:
cv_metrics = run_cv_baseline(df)
print(f"Accuracy: {np.mean(cv_metrics['accuracy']):.3f} ± {np.std(cv_metrics['accuracy']):.3f}")
print(f"Log-loss: {np.mean(cv_metrics['log_loss']):.3f} ± {np.std(cv_metrics['log_loss']):.3f}")
print(f"Brier:    {np.mean(cv_metrics['brier_score']):.3f} ± {np.std(cv_metrics['brier_score']):.3f}")
print(f"ECE:      {np.mean(cv_metrics['ece']):.3f} ± {np.std(cv_metrics['ece']):.3f}")


## Calibration Analysis

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from nba_predictor.evaluation.cv_strategy import playoff_season_cv_splits

Path("../reports/figures").mkdir(parents=True, exist_ok=True)

feat_cols = get_feature_cols(df)
all_probs, all_targets = [], []

for train_idx, test_idx in playoff_season_cv_splits(df):
    train, test = df.loc[train_idx], df.loc[test_idx]
    X_tr = train[feat_cols].fillna(0)
    y_tr = train["higher_seed_wins"].values
    X_te = test[feat_cols].fillna(0)
    y_te = test["higher_seed_wins"].values
    scaler = StandardScaler()
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(scaler.fit_transform(X_tr), y_tr)
    probs = lr.predict_proba(scaler.transform(X_te))[:, 1]
    all_probs.extend(probs.tolist())
    all_targets.extend(y_te.tolist())

all_probs   = np.array(all_probs)
all_targets = np.array(all_targets)

# Reliability diagram
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_mids  = (bin_edges[:-1] + bin_edges[1:]) / 2
frac_pos, mean_pred = [], []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (all_probs >= lo) & (all_probs < hi)
    if mask.sum() > 0:
        frac_pos.append(all_targets[mask].mean())
        mean_pred.append(all_probs[mask].mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Perfect calibration")
ax.plot(mean_pred, frac_pos, "o-", color="steelblue", lw=2, ms=8, label="Logistic Regression")
ax.fill_between(mean_pred, frac_pos, mean_pred,
                alpha=0.15, color="steelblue", label="Calibration gap")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives (actual win rate)")
ax.set_title("Reliability Diagram — Baseline Logistic Regression")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

ax2 = axes[1]
ax2.hist(all_probs[all_targets == 1], bins=20, alpha=0.6, color="steelblue",
         label="Higher seed wins", density=True)
ax2.hist(all_probs[all_targets == 0], bins=20, alpha=0.6, color="tomato",
         label="Upset", density=True)
ax2.axvline(0.5, color="black", lw=1, ls="--")
ax2.set_xlabel("Predicted P(higher seed wins)")
ax2.set_title("Predicted Probability Distribution by Outcome")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/figures/06_baseline_calibration.png", dpi=120, bbox_inches="tight")
plt.show()

ece = np.mean([abs(fp - mp) for fp, mp in zip(frac_pos, mean_pred)])
print(f"Expected Calibration Error (ECE): {ece:.4f}")
print(f"OOF Accuracy: {(all_probs.round() == all_targets).mean():.3f}")
